# Qwen3-TTS audio renderer v2 (jelly-jlpt): exam clips from the kana readings, lesson dialogues from natural text

**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.

- **Upload:** `tools/audio/render-input.json` (made by `node tools/export-tracks.js`). The upload cell asks for it.
- **What it renders:** every unique clip, spoken from each line's `say`. Exam tracks: the authored kana reading.
  Lesson-dialogue lines: natural adult Japanese in full kanji (words the learner sees in kana written in their normal kanji spelling,
  particles and okurigana in kana, phrase spaces removed, numbers as their kana; `dialogueSpeech` in lib.js),
  and an `arch` (a character id from `tools/audio/cast.json`, sent as `characters`): ONE locked voice per character for all their lines,
  from that character's `voice` prompt. No per-line tone: delivery comes from punctuation (……, ！, ？).
  Clips listed in the file's `have` (already in the repo's `audio/`) and clips already in the output folder are skipped.
  The upload cell prints the live counts (to render, reused, already in the output folder).
- **Re-rendering without deleting anything:** a clip's name holds its character and text, so a changed text renders anew by itself.
  After changing a character's `voice` prompt, bump its `rev` in `tools/audio/cast.json` (2, 3, ...): its clips are keyed
  `<id>@<rev>` and all re-render. One bad clip (wrong voice, broken file): give that line `take: 2` (3, ...) in `data/n5/listening.js`:
  only that clip gets a new name (`#<take>` in its key). Re-export, upload, Run all.
- **Voices:** the preview cell plays one line per character. To iterate on a prompt, use **Tweak a voice**; **A/B text test** compares
  three ways of writing the same dialogue lines (listening only, writes nothing).
- **Resumable:** output goes to Google Drive (`MyDrive/jelly-audio-render`) when you allow the mount, else `/content/jelly-audio-render`
  (lost on disconnect). Each clip is written to a temp file and renamed, so a disconnect never leaves a half clip.
  After a disconnect, reconnect and **Run all** again: it picks up where it stopped. Later small deltas: re-export, upload the new file, Run all.
- **What comes back:** `jelly-audio-new.zip` = the newly rendered `<12 hex>.mp3` clips plus `manifest.json`
  (full per-track file list, reused clips included).

**Then, in the repo:**
1. Unzip into a folder, e.g. `.scratch/render/`.
2. Copy its mp3s into `audio/`.
3. `node tools/build-audio-manifest.js .scratch/render` (checks every name and file, writes `audio/manifest.js`).
4. `node tools/build-sw.js`
5. `node .claude/hooks/run-tests.js`
6. Old clips no longer named by the manifest can be deleted from `audio/` (then rerun step 4).

In [ ]:
!pip install -q -U qwen-tts soundfile tqdm

## Model and voice helpers (same as the v1 notebook)

In [ ]:
import torch, gc, json, numpy as np, soundfile as sf
from IPython.display import Audio, display
from google.colab import files as files_mod
from qwen_tts import Qwen3TTSModel
print(torch.cuda.get_device_name(0))

def load(name):  # T4: float16 (no bfloat16), no flash-attn
    return Qwen3TTSModel.from_pretrained(name, device_map="cuda:0", dtype=torch.float16)

_models = {}
def model(kind):  # loaded on first use
    ids = {"custom": "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice",
           "design": "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign",
           "base":   "Qwen/Qwen3-TTS-12Hz-1.7B-Base"}
    if kind not in _models: _models[kind] = load(ids[kind])
    return _models[kind]

def preset(speaker, instruct=None): return {"kind": "preset", "speaker": speaker, "instruct": instruct}
def design(description):            return {"kind": "design", "description": description}

def raw(spec, text, seed):
    torch.manual_seed(seed)
    if spec["kind"] == "preset":
        kw = {"instruct": spec["instruct"]} if spec.get("instruct") else {}
        w, sr = model("custom").generate_custom_voice(text=text, language="Japanese", speaker=spec["speaker"], **kw)
    else:
        w, sr = model("design").generate_voice_design(text=text, language="Japanese", instruct=spec["description"])
    return np.asarray(w[0], dtype=np.float32), sr

def locked_speaker(spec, ref_text, seed):
    """speak(text, seed) -> (wave, sr), always through a clone of one reference line, so the voice cannot drift."""
    ref_wave, sr = raw(spec, ref_text, seed)
    prompt = model("base").create_voice_clone_prompt(ref_audio=(ref_wave, sr), ref_text=ref_text)
    def speak(text, s):
        torch.manual_seed(s)
        w, sr2 = model("base").generate_voice_clone(text=text, language="Japanese", voice_clone_prompt=prompt)
        return np.asarray(w[0], dtype=np.float32), sr2
    return speak

def loudness(w, target_db):
    """Scale a line so its speech (silence ignored) sits at target_db RMS; never clip."""
    active = w[np.abs(w) > 0.02 * np.abs(w).max()]
    g = 10 ** (target_db / 20) / (np.sqrt(np.mean(active ** 2)) + 1e-9)
    w = w * g
    peak = np.abs(w).max()
    return w * (0.97 / peak) if peak > 0.97 else w

## Output folder (Google Drive, so a disconnect resumes)
Allow the Drive prompt. If you skip it, clips go to `/content` and are lost on disconnect.

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/jelly-audio-render"
except Exception as e:
    print("Drive not mounted (", e, "): using /content, NOT resumable after a disconnect")
    OUT = "/content/jelly-audio-render"
os.makedirs(OUT, exist_ok=True)
for f in os.listdir(OUT):          # leftovers of an interrupted write
    if f.endswith(".tmp"): os.remove(os.path.join(OUT, f))
print("output:", OUT)

## Settings (the values you approved) and the input file

In [ ]:
import hashlib, zlib
SEED      = 1234
TARGET_DB = -20
GAIN_DB   = {"F": 1, "M": 0, "N": 0}

# Fixed reference sentence every voice is cloned from. Same for all characters.
REF_TEXT = "おはようございます。今日は天気がいいので、駅まで歩いて行きます。"

NARRATOR = preset("Serena", "Speak slowly and clearly in a calm, neutral tone like an exam recording.")

print("Upload tools/audio/render-input.json")
up = files_mod.upload()
DATA = json.loads(next(iter(up.values())).decode("utf-8"))
TRACKS, MAN, WOMAN = DATA["tracks"], DATA["manArchetypes"], DATA["womanArchetypes"]
CHARS = DATA.get("characters", {})          # dialogue characters: id -> { name, gender, voice, ... }
HAVE = set(DATA.get("have", []))

def arch_of(track, line):                    # a dialogue line names its character; the rest use the track's man / woman archetype
    return line.get("arch") or {"M": track["man"], "F": track["woman"]}.get(line["role"]) or ""

def clip_key(role, arch, say, take=None):     # same rule as tools/build-audio-manifest.js; arch '<id>@<rev>' = a revised voice
    return role + "|" + ("" if role == "N" else arch) + "|" + say + ("#" + str(take) if take and take > 1 else "")  # '#<take>' = a re-render

def clip_name(key):
    return hashlib.sha1(key.encode("utf-8")).hexdigest()[:12] + ".mp3"

CLIPS = {}                                    # name -> (role, arch, say, key)
for t in TRACKS:
    for l in t["lines"]:
        a = arch_of(t, l); k = clip_key(l["role"], a, l["say"], l.get("take"))
        CLIPS.setdefault(clip_name(k), (l["role"], a, l["say"], k))
from tqdm.auto import tqdm
SPEAKERS = {}
def speaker(role, arch):                      # built on first use: a small delta loads only the voices it needs
    if (role, arch) not in SPEAKERS:
        if role == "N": spec = NARRATOR
        elif arch.split("@")[0] in CHARS: spec = design(CHARS[arch.split("@")[0]]["voice"])   # a dialogue character ('<id>@<rev>' = same id, revised prompt): one locked designed voice for all their lines
        elif role == "M": spec = design(MAN[arch])
        else: spec = preset("Ono_Anna", WOMAN[arch])
        SPEAKERS[(role, arch)] = locked_speaker(spec, REF_TEXT, SEED)
    return SPEAKERS[(role, arch)]

on_disk = lambda: {f for f in os.listdir(OUT) if f.endswith(".mp3")}
TODO = sorted(n for n in CLIPS if n not in HAVE and n not in on_disk())
print(len(TRACKS), "tracks,", len(CLIPS), "unique clips:", len(TODO), "to render,",
      len([n for n in CLIPS if n in HAVE]), "reused from audio/,",
      len([n for n in CLIPS if n not in HAVE and n in on_disk()]), "already in the output folder")

## Preview (optional): one line per character that still has clips to render
Listen before the full render. Skip this cell if the voices are already approved.

In [ ]:
LABEL = {"N": "Narrator", "M": "Man", "F": "Woman"}
seen = set()
for n in TODO:
    role, arch, say, key = CLIPS[n]
    if (role, arch) in seen or len(say) <= 6: continue
    seen.add((role, arch))
    w, sr = speaker(role, arch)(say, SEED)
    print(LABEL[role], "|", arch or "fixed", "|", say)
    display(Audio(loudness(w, TARGET_DB + GAIN_DB.get(role, 0)), rate=sr))

## Tweak a voice (optional)
Try a new prompt for one character and hear one line. Copy the prompt you keep into `tools/audio/cast.json` and bump that character's `rev` (2, 3, ...): its clips get new names and re-render. Delete nothing.

In [ ]:
# Iterate on one character's voice without re-running everything. Edit CHAR / VOICE / LINE, run this cell, listen, repeat.
# Once happy: copy VOICE into that character's "voice" in tools/audio/cast.json, BUMP its "rev" (no rev = 1, so set 2, then 3, ...),
# and re-export render-input.json. The rev is part of the clip names ('<id>@<rev>'), so all that character's clips re-render
# under new names: delete nothing. Old clips are just no longer in the manifest.
CHAR  = "sasuke"
VOICE = CHARS[CHAR]["voice"]   # replace with the new prompt, e.g. "Native Japanese speaker, ... Clearly male young man about 17 ..."
LINE  = "……学生です。"          # any dialogue line, written as the render speaks it (see the A/B cell)
CHARS[CHAR]["voice"] = VOICE
for k in [k for k in SPEAKERS if k[1].split("@")[0] == CHAR]: del SPEAKERS[k]   # drop the cached locked voice (any rev) so the new prompt is used
w, sr = speaker(CHARS[CHAR]["gender"], CHAR)(LINE, SEED)
print(CHAR, "|", VOICE, "|", LINE)
display(Audio(loudness(w, TARGET_DB + GAIN_DB.get(CHARS[CHAR]["gender"], 0)), rate=sr))

## A/B text test (optional)
The same three dialogue lines spoken three ways: A = the speech text the render uses (full kanji), B = the display text with phrase spaces, C = the old all-kana style. Listening only.

In [ ]:
# The same three dialogue lines in three spellings, each in its character's voice. Listening only: writes no clips.
#   A = the speech text the render uses (full kanji: kana words in their kanji spelling, phrase spaces removed, numbers as kana)
#   B = the display text WITH the full-width phrase spaces
#   C = old style: all kana, spaces stripped (with the old particle swap は -> わ, …… dropped)
AB = [  # (character, A, B, C): a particle は, a number, a 「……」 line
    ("lelouch", "いいえ。私は学生です。", "いいえ。わたしは　学生です。", "いいえ。わたしわがくせいです。"),
    ("hinata", "はい！タクシーはいちまんさんぜんえんでした！", "はい！　タクシーは　一まん三ぜん円でした！", "はい！タクシーわいちまんさんぜんえんでした！"),
    ("sasuke", "……学生です。", "……学生です。", "がくせいです。"),
]
for who, *texts in AB:
    for label, text in zip("ABC", texts):
        w, sr = speaker(CHARS[who]["gender"], who)(text, SEED)
        print(label, "|", who, "|", text)
        display(Audio(loudness(w, TARGET_DB + GAIN_DB.get(CHARS[who]["gender"], 0)), rate=sr))

## Render
Safe to stop and re-run: finished clips are skipped, each clip is written atomically.

In [ ]:
import subprocess
TODO = sorted(n for n in CLIPS if n not in HAVE and n not in on_disk())
for n in tqdm(TODO, desc="clips"):
    role, arch, say, key = CLIPS[n]
    w, sr = speaker(role, arch)(say, SEED + zlib.crc32(key.encode("utf-8")) % 100000)
    w = loudness(w, TARGET_DB + GAIN_DB.get(role, 0))
    sf.write("/content/_tmp.wav", w, sr)
    tmp = os.path.join(OUT, n + ".tmp")
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", "/content/_tmp.wav", "-ac", "1", "-ar", "24000", "-b:a", "48k", "-f", "mp3", tmp], check=True)
    os.replace(tmp, os.path.join(OUT, n))
print(len(TODO), "rendered")

## Verify and write manifest.json

In [ ]:
missing = [n for n in CLIPS if n not in HAVE and n not in on_disk()]
assert not missing, f"{len(missing)} clips missing, re-run the Render cell: {missing[:5]}"
manifest = {"version": 1, "tracks": {t["id"]: [clip_name(clip_key(l["role"], arch_of(t, l), l["say"], l.get("take"))) for l in t["lines"]] for t in TRACKS}}
assert all(len(manifest["tracks"][t["id"]]) == len(t["lines"]) for t in TRACKS)
tmp = os.path.join(OUT, "manifest.json.tmp")
json.dump(manifest, open(tmp, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
os.replace(tmp, os.path.join(OUT, "manifest.json"))
NEW = sorted(n for n in CLIPS if n not in HAVE)
print("OK:", len(CLIPS), "clips accounted for,", len(NEW), "new in", OUT, "+ manifest.json")

## Zip and download (new clips + manifest.json only)

In [ ]:
import zipfile
ZIP = "/content/jelly-audio-new.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_STORED) as z:   # mp3 does not compress
    for n in NEW: z.write(os.path.join(OUT, n), n)
    z.write(os.path.join(OUT, "manifest.json"), "manifest.json")
print(f"{len(NEW)} clips, {os.path.getsize(ZIP) / 1e6:.1f} MB")
files_mod.download(ZIP)